In [1]:
from pathlib import Path

folder = Path.cwd()
print("Working directory:", folder)
print("Files:", [p.name for p in folder.iterdir()])

Working directory: C:\Users\farzan\Dropbox\PC\Desktop\CSC6410_Experiment3
Files: ['.ipynb_checkpoints', 'anaconda_projects', 'CSC6410_Experiment3.ipynb', 'host_model.yang', 'host_schema.json', 'intended_example.json', 'switch_schema.json', 'validate_config.py']


In [2]:
import json
from pathlib import Path

with open("intended_example.json", "r", encoding="utf-8") as f:
    example = json.load(f)

print(json.dumps(example, indent=4))

{
    "student-host:host": {
        "os": "windows",
        "default-gateway": "192.168.1.1",
        "dns-server": [
            "192.168.1.1"
        ],
        "interface": [
            {
                "name": "Wi-Fi",
                "type": "wifi",
                "oper-status": "up",
                "mtu": 1500
            },
            {
                "name": "Ethernet",
                "type": "ethernet",
                "oper-status": "down"
            }
        ]
    }
}


In [3]:
intended = {
    "student-host:host": {
        "hostname": "DESKTOP-EXAMPLE",
        "os": "windows",
        "default-gateway": "192.168.1.1",
        "dns-server": ["192.168.1.1"],
        "interface": [
            {
                "name": "Wi-Fi",
                "type": "wifi",
                "oper-status": "up",
                "mtu": 1500
            },
            {
                "name": "Ethernet",
                "type": "ethernet",
                "oper-status": "down"
            }
        ]
    }
}

with open("intended.json", "w", encoding="utf-8") as f:
    json.dump(intended, f, indent=4)

print("intended.json created successfully")

intended.json created successfully


In [4]:
from jsonschema import Draft202012Validator

with open("host_schema.json", "r", encoding="utf-8") as f:
    schema = json.load(f)

with open("intended.json", "r", encoding="utf-8") as f:
    intended = json.load(f)

validator = Draft202012Validator(schema)
errors = list(validator.iter_errors(intended))

if not errors:
    print("VALID: intended.json matches the course schema")
else:
    for error in errors:
        path = "/".join(map(str, error.absolute_path))
        print(f"ERROR at {path}: {error.message}")

VALID: intended.json matches the course schema


In [5]:
import json
from pathlib import Path

intended = {
    "student-host:host": {
        "os": "windows",
        "default-gateway": "192.168.12.1",
        "dns-server": [
            "192.168.12.1"
        ],
        "interface": [
            {
                "name": "Wi-Fi",
                "type": "wifi",
                "oper-status": "up",
                "mtu": 1500
            },
            {
                "name": "Ethernet 2",
                "type": "virtual",
                "oper-status": "up",
                "mtu": 1500
            },
            {
                "name": "NordLynx",
                "type": "virtual",
                "oper-status": "down",
                "mtu": 1350
            }
        ]
    }
}

Path("intended.json").write_text(
    json.dumps(intended, indent=4),
    encoding="utf-8"
)

print("Updated intended.json successfully")
print(json.dumps(intended, indent=4))

Updated intended.json successfully
{
    "student-host:host": {
        "os": "windows",
        "default-gateway": "192.168.12.1",
        "dns-server": [
            "192.168.12.1"
        ],
        "interface": [
            {
                "name": "Wi-Fi",
                "type": "wifi",
                "oper-status": "up",
                "mtu": 1500
            },
            {
                "name": "Ethernet 2",
                "type": "virtual",
                "oper-status": "up",
                "mtu": 1500
            },
            {
                "name": "NordLynx",
                "type": "virtual",
                "oper-status": "down",
                "mtu": 1350
            }
        ]
    }
}


In [6]:
%pip install jsonschema psutil

Note: you may need to restart the kernel to use updated packages.


In [7]:
from pathlib import Path

print(Path.cwd())

print("Schema exists:", Path("host_schema.json").exists())
print("Intended exists:", Path("intended.json").exists())

C:\Users\farzan\Dropbox\PC\Desktop\CSC6410_Experiment3
Schema exists: True
Intended exists: True


In [8]:

%%writefile network_config_checker.py
import json
import subprocess
import socket
import time
from datetime import datetime
from pathlib import Path
from jsonschema import Draft202012Validator

# Files are stored beside this Python script.
HERE = Path(__file__).resolve().parent
TOP = "student-host:host"


def powershell(command):
    # Run a Windows command and receive structured JSON.
    result = subprocess.run(
        [
            "powershell.exe",
            "-NoProfile",
            "-Command",
            command
        ],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=True
    )

    output = result.stdout.strip()
    return json.loads(output) if output else []


def collect_configuration():
    # Collect multiple Windows network data sources.
    adapters = powershell(
        "ConvertTo-Json -InputObject @("
        "Get-NetAdapter -IncludeHidden | "
        "Select-Object Name, InterfaceDescription, "
        "ifIndex, Status, MacAddress"
        ") -Depth 4"
    )

    addresses = powershell(
        "ConvertTo-Json -InputObject @("
        "Get-NetIPAddress | "
        "Select-Object InterfaceIndex, InterfaceAlias, "
        "IPAddress, PrefixLength"
        ") -Depth 4"
    )

    ip_interfaces = powershell(
        "ConvertTo-Json -InputObject @("
        "Get-NetIPInterface | "
        "Select-Object InterfaceIndex, InterfaceAlias, "
        "NlMtu, InterfaceMetric, "
        "@{Name='Family';Expression={$_.AddressFamily.ToString()}}, "
        "@{Name='State';Expression={$_.ConnectionState.ToString()}}"
        ") -Depth 4"
    )

    routes = powershell(
        "ConvertTo-Json -InputObject @("
        "Get-NetRoute -AddressFamily IPv4 | "
        "Where-Object {$_.DestinationPrefix -eq '0.0.0.0/0'} | "
        "Select-Object InterfaceIndex, NextHop, RouteMetric"
        ") -Depth 4"
    )

    dns = powershell(
        "ConvertTo-Json -InputObject @("
        "Get-DnsClientServerAddress | "
        "Select-Object InterfaceIndex, ServerAddresses"
        ") -Depth 4"
    )

    host = {
        "hostname": socket.gethostname(),
        "os": "windows",
        "collected-at": datetime.now().astimezone().isoformat(),
        "interface": []
    }

    # Match records using interface index, not list order.
    adapter_map = {
        a["ifIndex"]: a for a in adapters
    }

    address_map = {}
    for address in addresses:
        index = address["InterfaceIndex"]
        address_map.setdefault(index, []).append(address)

    ip_map = {}
    for row in ip_interfaces:
        index = row["InterfaceIndex"]
        ip_map.setdefault(index, []).append(row)

    # Include interfaces missing from Get-NetAdapter.
    all_indexes = (
        set(adapter_map)
        | set(address_map)
        | set(ip_map)
    )

    for index in sorted(all_indexes):
        adapter = adapter_map.get(index, {})
        ip_rows = ip_map.get(index, [])
        addr_rows = address_map.get(index, [])

        name = adapter.get("Name")

        if not name and ip_rows:
            name = ip_rows[0]["InterfaceAlias"]

        if not name and addr_rows:
            name = addr_rows[0]["InterfaceAlias"]

        if not name:
            continue

        description = (
            name + " " +
            str(adapter.get("InterfaceDescription") or "")
        ).lower()

        # Classify physical and virtual interfaces.
        if "loopback" in description:
            interface_type = "loopback"

        elif any(word in description for word in [
            "virtual", "vethernet", "nordlynx",
            "vpn", "hyper-v", "tap-", "tunnel"
        ]):
            interface_type = "virtual"

        elif "wi-fi" in description or "wireless" in description:
            interface_type = "wifi"

        elif "bluetooth" in description:
            interface_type = "other"

        elif "ethernet" in description:
            interface_type = "ethernet"

        else:
            interface_type = "other"

        # Convert Windows status into the model's up/down values.
        status = str(adapter.get("Status") or "").lower()

        if status:
            is_up = status == "up"
        else:
            is_up = any(
                str(row["State"]).lower() == "connected"
                for row in ip_rows
            )

        interface = {
            "name": name,
            "type": interface_type,
            "oper-status": "up" if is_up else "down",
            "ipv4": [],
            "ipv6": []
        }

        # Convert MAC addresses to lowercase colon format.
        mac = adapter.get("MacAddress")

        if mac:
            normalized = mac.lower().replace("-", ":")
            interface["mac"] = normalized

        # Prefer IPv4 MTU because the schema has one MTU field.
        selected = next(
            (row for row in ip_rows
             if row["Family"] == "IPv4"),
            ip_rows[0] if ip_rows else None
        )

        if selected:
            mtu = selected.get("NlMtu")

            # Windows loopback can exceed the course model's limit.
            # Keep the interface but omit an unrepresentable MTU.
            if isinstance(mtu, int) and 68 <= mtu <= 65536:
                interface["mtu"] = mtu
            else:
                print(
                    f"NOTE: {name}: MTU {mtu} "
                    "is outside the course schema range."
                )

        # Include prefix lengths and remove IPv6 scope identifiers.
        for row in addr_rows:
            ip = row["IPAddress"].split("%")[0]
            prefix = row["PrefixLength"]

            family = "ipv6" if ":" in ip else "ipv4"
            value = f"{ip}/{prefix}"

            if value not in interface[family]:
                interface[family].append(value)

        host["interface"].append(interface)

    # The model stores one default gateway.
    # Prefer routes with the lowest route + interface metric.
    metrics = {
        row["InterfaceIndex"]: row["InterfaceMetric"]
        for row in ip_interfaces
        if row["Family"] == "IPv4"
    }

    usable_routes = [
        route for route in routes
        if route.get("NextHop") not in (None, "", "0.0.0.0")
    ]

    if usable_routes:
        best_route = min(
            usable_routes,
            key=lambda route: (
                route["RouteMetric"]
                + metrics.get(route["InterfaceIndex"], 0)
            )
        )

        host["default-gateway"] = best_route["NextHop"]

        # Use DNS servers belonging to the selected route's interface.
        main_index = best_route["InterfaceIndex"]

        host["dns-server"] = list(dict.fromkeys(
            server
            for row in dns
            if row["InterfaceIndex"] == main_index
            for server in (row.get("ServerAddresses") or [])
        ))

    else:
        # A disconnected computer might not have a default route.
        host["dns-server"] = []
        print("NOTE: No IPv4 default gateway found.")

    return {TOP: host}


def validate_configuration(observed):
    # Validate actual collected data using the professor's schema.
    schema = json.loads(
        (HERE / "host_schema.json").read_text(encoding="utf-8")
    )

    validator = Draft202012Validator(schema)
    errors = list(validator.iter_errors(observed))

    print("\n=== SCHEMA VALIDATION ===")

    if not errors:
        print("VALID: observed.json matches host_schema.json")
    else:
        print(f"INVALID: {len(errors)} error(s)")

        for error in errors:
            path = "/".join(
                str(part) for part in error.absolute_path
            )

            print(f"ERROR at {path}: {error.message}")

    return errors


def report_drift(intended, observed):
    # Compare only the fields specified in intended.json.
    expected = intended[TOP]
    actual = observed[TOP]
    differences = []

    for field, value in expected.items():
        if field == "interface":
            continue

        observed_value = actual.get(field, "<missing>")

        if value != observed_value:
            differences.append(
                f"{field}: intended={value}, "
                f"observed={observed_value}"
            )

    # Match interfaces by name, not by their position in JSON.
    actual_interfaces = {
        interface["name"]: interface
        for interface in actual.get("interface", [])
    }

    for expected_interface in expected.get("interface", []):
        name = expected_interface["name"]
        actual_interface = actual_interfaces.get(name)

        if actual_interface is None:
            differences.append(
                f"interface {name}: intended=present, "
                "observed=missing"
            )
            continue

        for field, value in expected_interface.items():
            if field == "name":
                continue

            observed_value = actual_interface.get(
                field, "<missing>"
            )

            if value != observed_value:
                differences.append(
                    f"interface {name}/{field}: "
                    f"intended={value}, "
                    f"observed={observed_value}"
                )

    print("\n=== CONFIGURATION DRIFT ===")

    if differences:
        for difference in differences:
            print(difference)
    else:
        print("No drift detected.")

    print("Total differences:", len(differences))

    return differences


def main():
    start = time.perf_counter()

    observed = collect_configuration()

    (HERE / "observed.json").write_text(
        json.dumps(observed, indent=4),
        encoding="utf-8"
    )

    elapsed = time.perf_counter() - start

    print("\n=== COLLECTION RESULTS ===")
    print("Observed configuration saved to observed.json")
    print("Hostname:", observed[TOP]["hostname"])
    print("Interfaces:", len(observed[TOP]["interface"]))
    print("Collection time:", round(elapsed, 3), "seconds")

    errors = validate_configuration(observed)

    intended = json.loads(
        (HERE / "intended.json").read_text(encoding="utf-8")
    )

    differences = report_drift(intended, observed)

    summary = {
        "collection_seconds": elapsed,
        "interface_count": len(observed[TOP]["interface"]),
        "schema_error_count": len(errors),
        "drift_count": len(differences),
        "differences": differences
    }

    (HERE / "results.json").write_text(
        json.dumps(summary, indent=4),
        encoding="utf-8"
    )

    return observed, summary


if __name__ == "__main__":
    main()


Writing network_config_checker.py


In [9]:
%run network_config_checker.py

NOTE: Loopback Pseudo-Interface 1: MTU 4294967295 is outside the course schema range.

=== COLLECTION RESULTS ===
Observed configuration saved to observed.json
Hostname: DESKTOP-S75CK0H
Interfaces: 23
Collection time: 10.035 seconds

=== SCHEMA VALIDATION ===
VALID: observed.json matches host_schema.json

=== CONFIGURATION DRIFT ===
No drift detected.
Total differences: 0


In [10]:
import json

with open("observed.json", "r", encoding="utf-8") as f:
    observed = json.load(f)

print(json.dumps(observed, indent=4))

{
    "student-host:host": {
        "hostname": "DESKTOP-S75CK0H",
        "os": "windows",
        "collected-at": "2026-11-12T17:49:52.286912-06:00",
        "interface": [
            {
                "name": "Loopback Pseudo-Interface 1",
                "type": "loopback",
                "oper-status": "up",
                "ipv4": [
                    "127.0.0.1/8"
                ],
                "ipv6": [
                    "::1/128"
                ]
            },
            {
                "name": "Ethernet 2",
                "type": "virtual",
                "oper-status": "up",
                "ipv4": [
                    "192.168.56.1/24"
                ],
                "ipv6": [
                    "fe80::b37b:57a5:5126:ccb8/64"
                ],
                "mac": "0a:00:27:00:00:02",
                "mtu": 1500
            },
            {
                "name": "6to4 Adapter",
                "type": "other",
                "oper-status": "down"

In [11]:
import pandas as pd

interfaces = observed["student-host:host"]["interface"]

df = pd.DataFrame(interfaces)

df[["name", "type", "oper-status", "mtu", "ipv4", "ipv6"]]

,name,type,oper-status,mtu,ipv4,ipv6
0,Loopback Pseudo-Interface 1,loopback,up,NaN,[127.0.0.1/8],[::1/128]
1,Ethernet 2,virtual,up,1500.0,[192.168.56.1/24],[fe80::b37b:57a5:5126:ccb8/64]
2,6to4 Adapter,other,down,NaN,[],[]
3,Bluetooth Network Connection,other,down,1500.0,[169.254.252.192/16],[fe80::f9ff:615d:9930:a012/64]
4,Local Area Connection* 8,other,up,NaN,[],[]
5,Microsoft IP-HTTPS Platform Interface,other,down,NaN,[],[]
6,Ethernet (Kernel Debugger),ethernet,down,NaN,[],[]
7,Wi-Fi,wifi,up,1500.0,[192.168.12.14/24],[]
8,Local Area Connection* 9,other,up,NaN,[],[]
9,Local Area Connection* 6,other,down,NaN,[],[]


In [12]:
import json
import copy
from pathlib import Path
from jsonschema import Draft202012Validator

schema = json.loads(
    Path("host_schema.json").read_text()
)

observed = json.loads(
    Path("observed.json").read_text()
)

# Make a copy so the original configuration stays unchanged.
broken = copy.deepcopy(observed)

# Reinsert the actual Windows loopback MTU.
for interface in broken["student-host:host"]["interface"]:
    if interface["type"] == "loopback":
        interface["mtu"] = 4294967295
        break

validator = Draft202012Validator(schema)

errors = list(validator.iter_errors(broken))

print("=== LOOPBACK MTU VALIDATION TEST ===")

for error in errors:
    path = "/".join(map(str, error.absolute_path))
    print("Error path:", path)
    print("Error:", error.message)

print("Total errors:", len(errors))

Path("observed_invalid_test.json").write_text(
    json.dumps(broken, indent=4)
)

=== LOOPBACK MTU VALIDATION TEST ===
Error path: student-host:host/interface/0/mtu
Error: 4294967295 is greater than the maximum of 65536
Total errors: 1


6879

In [13]:
errors = list(validator.iter_errors(observed))

print("=== CORRECTED CONFIGURATION ===")

if not errors:
    print("VALID: Original observed.json passes validation.")
else:
    for error in errors:
        print(error.message)

print("Total schema errors:", len(errors))

=== CORRECTED CONFIGURATION ===
VALID: Original observed.json passes validation.
Total schema errors: 0


In [14]:
import subprocess

command = """
Get-NetIPInterface -InterfaceAlias 'NordLynx' |
Select-Object InterfaceAlias, AddressFamily, NlMtu |
Format-Table -AutoSize
"""

result = subprocess.run(
    ["powershell.exe", "-NoProfile", "-Command", command],
    capture_output=True,
    text=True,
    check=True
)

print("=== ORIGINAL WINDOWS MTU VALUES ===")
print(result.stdout)

print("=== MTU STORED IN OBSERVED.JSON ===")

for interface in observed["student-host:host"]["interface"]:
    if interface["name"] == "NordLynx":
        print("Interface:", interface["name"])
        print("Selected MTU:", interface.get("mtu"))

=== ORIGINAL WINDOWS MTU VALUES ===

InterfaceAlias AddressFamily NlMtu
-------------- ------------- -----
NordLynx                IPv6  1420
NordLynx                IPv4  1350



=== MTU STORED IN OBSERVED.JSON ===
Interface: NordLynx
Selected MTU: 1350


In [15]:
import shutil

shutil.copyfile(
    "observed.json",
    "observed_before.json"
)

shutil.copyfile(
    "results.json",
    "results_before.json"
)

print("Baseline configuration saved.")

Baseline configuration saved.


In [16]:
%run network_config_checker.py

NOTE: Loopback Pseudo-Interface 1: MTU 4294967295 is outside the course schema range.

=== COLLECTION RESULTS ===
Observed configuration saved to observed.json
Hostname: DESKTOP-S75CK0H
Interfaces: 23
Collection time: 10.243 seconds

=== SCHEMA VALIDATION ===
VALID: observed.json matches host_schema.json

=== CONFIGURATION DRIFT ===
interface Wi-Fi/oper-status: intended=up, observed=down
Total differences: 1


In [17]:
shutil.copyfile(
    "observed.json",
    "observed_after.json"
)

shutil.copyfile(
    "results.json",
    "results_after.json"
)

print("Changed configuration saved.")

Changed configuration saved.


In [18]:
shutil.copyfile(
    "observed.json",
    "observed_after.json"
)

shutil.copyfile(
    "results.json",
    "results_after.json"
)

print("Changed configuration saved.")

Changed configuration saved.


In [19]:
%run network_config_checker.py

NOTE: Loopback Pseudo-Interface 1: MTU 4294967295 is outside the course schema range.

=== COLLECTION RESULTS ===
Observed configuration saved to observed.json
Hostname: DESKTOP-S75CK0H
Interfaces: 23
Collection time: 10.145 seconds

=== SCHEMA VALIDATION ===
VALID: observed.json matches host_schema.json

=== CONFIGURATION DRIFT ===
No drift detected.
Total differences: 0


In [20]:
shutil.copyfile(
    "observed.json",
    "observed_restored.json"
)

shutil.copyfile(
    "results.json",
    "results_restored.json"
)

print("Restored configuration saved.")

Restored configuration saved.
